<h1>Chapter 8 - Multi-Agent Collaboration</h1>
<i>More Agents?!</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 8 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

# Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [2]:
import os
from illustrated_agents.llm import LLM

# Ollama
llm = LLM(model="ollama/gemma3:12b")

# Llama.cpp server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M", api_base="http://localhost:8080", api_key="sk-no-key-required")

# Llama-cpp-python server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M.gguf", api_base="http://localhost:8000/v1/", api_key="sk-no-key-required")

# LM Studio
# llm = LLM(model="lm_studio/gemma-3-12b-it", api_base="http://localhost:1234/v1", api_key="sk-no-key-required")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_GEMINI_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash")
# llm = LLM(model="gemini/gemma-3-12b-it")

# Agents as Tools

Agent orchestration can quickly be a daunting task, especially as you increasingly add many sub-Agents. There is a fortunately a nice trick to add collaboration amongst Agents using everything we have already created! The title gives it away, Agents as Tools. As covered in the book, there are various patterns that collaboration can take, but we focus on a central pattern where there is one Agent instructing a bunch of others. 

To run Agents as Tools, we first need to define them. So let's start by creating a `MathAgent`:

In [25]:
from illustrated_agents import Tools, Memory
from illustrated_agents.chapters.ch5 import TinyAgent as ToolAgent

def add(a: str, b: str) -> float:
    return float(a) + float(b)

def subtract(a: str, b: str) -> float:
    return float(a) - float(b)

def multiply(a: str, b: str) -> float:
    return float(a) * float(b)

# Math specialist
math_tools = Tools()
math_tools.add_tool("add", add, "Adds two numbers: add(a, b)")
math_tools.add_tool("subtract", subtract, "Subtracts two numbers: subtract(a, b)")
math_tools.add_tool("multiply", multiply, "Multiplies two numbers: multiply(a, b)")
memory = Memory()
math_agent = ToolAgent(llm=llm, tools=math_tools, memory=memory)

Note that we use the `TinyAgent` from chapter 5 which can only run a tool once and has no ReAct or Reflection. Next, we can convert the `math_agent` to a tool:

In [26]:
def ask_math_agent(question: str) -> str:
    """Delegate to the math specialist."""
    return math_agent.run(question)

We call the tool `ask_math_agent` so that your `TinyAgent` knows that the question is being rerouted to another agent. Now, let's create your `TinyAgent` with ReAct but without Reflection to make it more efficient:

In [27]:
from illustrated_agents import ReAct
from illustrated_agents.chapters.ch6a import TinyAgent

# Define tools
def get_weather(location: str) -> str:
    return f"Weather in {location}: Sunny, 72°F"

# Register tools
tools = Tools()
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(city)")
tools.add_tool("ask_math_agent", ask_math_agent, "Asks the math specialist: ask_math_agent(question)")

# Memory
memory = Memory()

# ReAct
react = ReAct(max_steps=10)

# Create agent
orchestrator_agent = TinyAgent(llm=llm, tools=tools, memory=memory, planner=react)

We named your `TinyAgent` the `orchestrator_agent` since it is in charge of using tools (including the `ask_math_agent` tool). We also use the same LLM for both Agents as that is easier since we do not have to run both of them on the samen device. In practice, however, you might want to use a smaller Agent as the `math_agent` since it only has to use tools. The `orchestrator_agent`, in contrast, tends to be a large model since it has to relay tasks and decide on the best action sequencing. 

Let's see if the `orchestrator_agent` you created works by asking it a basic question:

In [28]:
output = orchestrator_agent.run("What is 5.12 times 3.9?")
print(output)

19.968


The output is correct, but let's see if your `TinyAgent` actually used the `math_agent`:

In [29]:
orchestrator_agent.memory.get_messages()

[{'role': 'user',
  'content': 'You are a helpful AI agent.\n\n\n# ReACT (Reason and Act)\n\nYou are a ReAct agent that performs exactly ONE step per turn.\nMake sure to break down a given task into smaller steps and decide whether to use a tool or provide a final answer.\n\n## ReACT Format\n\nYou use the following format for each step:\n\nTHOUGHT: [Your reasoning about what to do next]\nACTION:\n{\n    "tool": "a_tool_name",\n    "args": [...],\n}\n\nIf no tool is needed and you want to provide an intermediate answer, use:\n\nACTION:\n{\n    "tool": "intermediate_answer",\n    "args": "insert your intermediate answer here"\n}\n\nAn observation will be provided after each action. You do not generate the observation yourself.\n\n## ReACT Completion\n\nTo provide the final answer to the task, use an action blob with "tool": "final_answer" tool. \nIt is the only way to complete the task, else you will be stuck on a loop. So your final output should look like this:\n\nACTION:\n{\n    "tool

We can even explore the memory of the `math_agent`:

In [32]:
math_agent.memory.get_messages()

[{'role': 'user',
  'content': 'You are a helpful assistant.\n\n\n# Tools\n\nIf needed, you can only use the following tools to assist you in completing tasks:\n\n`add`: Adds two numbers: add(a, b)\n`subtract`: Subtracts two numbers: subtract(a, b)\n`multiply`: Multiplies two numbers: multiply(a, b)\n\nTo use a tool, respond with JSON: {"tool": "name", "args": [...]}\n'},
 {'role': 'user', 'content': 'What is 5.12 times 3.9?'},
 {'role': 'assistant',
  'content': '{"tool": "multiply", "args": [5.12, 3.9]}\n'}]

Note that there is a "user" that asks "What is 5.12 times 3.9?". This is actually the `orchestrator_agent` that asked the question!